# Sistemas de Recomendación

In [1]:
import pandas as pd
import numpy as np

## Similitud coseno

$$sim(\pmb x, \pmb y) = \frac {\pmb x \cdot \pmb y}{||\pmb x|| \cdot ||\pmb y||}$$

¿Cómo calcularla en Python?

Supongamos que tenemos la siguiente matriz:

|  	| Libro A 	| Libro B 	| Libro C 	|
|-------	|---------	|---------	|---------	|
| Juan 	| 5 	| 4 	| 4 	|
| Diego 	| 4 	| 5 	| 5 	|


Podemos calcular la similitud coseno empleando sklearn:

In [2]:
from sklearn.metrics.pairwise import cosine_similarity
Juan = [5,4,4]
Diego = [4,5,5]
cosine_similarity([Juan, Diego])

array([[1.        , 0.97823198],
       [0.97823198, 1.        ]])

También podemos calcular la similitud a mano:

In [3]:
(5*4 + 4*5 + 4*5)/(np.sqrt(5**2+4**2+4**2)*np.sqrt(4**2+5**2+5**2))

0.9782319760890369

O empleando Numpy

Calcular la similitud coseno usando numpy (con np.dot y np.linalg.norm)

In [4]:
np.dot(Juan,Diego)/np.dot(np.linalg.norm(Juan), np.linalg.norm(Diego))

0.9782319760890369

Ahora bien, cuando tenemos una matriz user-item de la vida real, tenemos muchos casos faltantes. En esta situación, no podremos calcular la similitud coseno tan fácilmente...

In [5]:
user_item = np.array([[5, np.nan, 4],[4,3,5],[4,5,5],[np.nan, 5, np.nan], [np.nan, 5, 3]])
user_item

array([[ 5., nan,  4.],
       [ 4.,  3.,  5.],
       [ 4.,  5.,  5.],
       [nan,  5., nan],
       [nan,  5.,  3.]])

## Surprise

En esta notebook vamos a emplear la librería surprise. Esta es una librería que se basa en la API de scikit-learn y permite implementar varios algoritmos básicos de recomendación.

Comencemos cargando un dataset clásico en sistemas de recomendación: MovieLens (https://movielens.org/). Esta es una página de recomendación de películas que abrió información histórica.

In [6]:
!pip list

Package                           Version
--------------------------------- ------------------
adagio                            0.2.6
aext-assistant                    4.20.0
aext-assistant-server             4.20.0
aext-core                         4.20.0
aext-core-server                  4.20.0
aext_environments_server          4.20.0
aext-panels                       4.20.0
aext-panels-server                4.20.0
aext-project-filebrowser-server   4.20.0
aext-share-notebook               4.20.0
aext-share-notebook-server        4.20.0
aext-shared                       4.20.0
aext-toolbox                      4.20.0
affine                            2.4.0
aiobotocore                       2.12.3
aiohappyeyeballs                  2.4.0
aiohttp                           3.10.5
aioitertools                      0.7.1
aiosignal                         1.2.0
alabaster                         0.7.16
alembic                           1.13.3
altair                            5.0.1
anaconda-

In [7]:
!pip install surprise
# https://surprise.readthedocs.io/

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-macosx_10_15_x86_64.whl size=493047 sha256=84a5f7b90593e72732ac4b04bd5d52773cbe306fffff3fb8c0c6b14679ba9501
  Stored in directory: /Users/abc/Library/Caches/pip/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise


In [8]:
# Bajamos el dataset. En windows pueden descargarlo entrando al link manualmente
#!wget https://files.grouplens.org/datasets/movielens/ml-100k/u.data

In [9]:
# información sobre Movie lens https://files.grouplens.org/datasets/movielens/ml-100k-README.txt

mlens = pd.read_csv("https://files.grouplens.org/datasets/movielens/ml-100k/u.data",sep="\t",header=None)
mlens.columns = ["user_id","item_id","rating","timestamp"]
mlens

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
...,...,...,...,...
99995,880,476,3,880175444
99996,716,204,5,879795543
99997,276,1090,1,874795795
99998,13,225,2,882399156


In [10]:
len(mlens['user_id'].unique())

943

In [11]:
len(mlens['item_id'].unique())

1682

In [12]:
mlens.drop("timestamp", axis=1, inplace=True)

In [13]:
mlens

,user_id,item_id,rating
0,196,242,3
1,186,302,3
2,22,377,1
3,244,51,2
4,166,346,1
...,...,...,...
99995,880,476,3
99996,716,204,5
99997,276,1090,1
99998,13,225,2


El paquete surprise no recibe directamente un objeto DataFrame.
Para parsear y leer un conjunto de datos debe hacerlo a través de dos nuevos objetos: Reader y Dataset.
En Reader debemos especificar el valor mínimo y el valor máximo de los ratings.
Dataset nos permite leer datos desde distintas fuentes.

In [ ]:
from surprise import Dataset, Reader

# Creamos un objeto Reader indicando el rango de las valoraciones (ratings).
# mlens["rating"].min() → obtiene el valor mínimo de la columna "rating"
# mlens["rating"].max() → obtiene el valor máximo de la columna "rating"
# Esto asegura que el sistema entienda cuál es el rango válido de puntuaciones.
reader = Reader(rating_scale=(mlens["rating"].min(), mlens["rating"].max()))

In [15]:
dataset = Dataset.load_from_df(mlens,reader)
dataset

Ahora cargue SVD y GridSearchCV, ambos de surprise.

In [ ]:
from surprise import SVD
from surprise.model_selection.search import GridSearchCV

"\nclasssurprise.prediction_algorithms.matrix_factorization.SVD(n_factors=100,\nn_epochs=20, biased=True, init_mean=0, init_std_dev=0.1, lr_all=0.005, reg_all=0.02,\nlr_bu=None, lr_bi=None, lr_pu=None, lr_qi=None, reg_bu=None, reg_bi=None,\nreg_pu=None, reg_qi=None, random_state=None, verbose=False)\n\nclass surprise.model_selection.search.GridSearchCV(algo_class, param_grid,\nmeasures=['rmse', 'mae'], cv=None, refit=False, return_train_measures=False,\nn_jobs=1, pre_dispatch='2*n_jobs', joblib_verbose=0)\n"

Genere una grilla de parámetros donde se prueben distintas combinaciones de:  
  - epochs: es la cantidad de pasadas sobre el dataset que hará el algoritmo empleando descenso por el gradiente  
  - biased: usar parámetros de sesgo o no  
  - lr_all: learning rate para todos los parámetros  
  - reg_all: término de regularización para todos los parámetros (lambda)  

In [17]:
param_grid = {'n_epochs': [5, 10], 'lr_all': [0.002, 0.005], 'reg_all': [0.4, 0.6]}

Emplee GridSearchCV, SVD y el diccionario con los parámetros para probar, y entrene un modelo. Note que a GridSearchCV necesita pasarle un modelo sin instanciar. Además, setee el parámetro refit a True y con measures = ["rmse","fcp"]

In [ ]:
# Definimos un diccionario con los hiperparámetros que vamos a probar en el modelo
param_grid = {
    'n_epochs': [5, 10],     # ⏳ Número de épocas (iteraciones de entrenamiento) a probar: 5 o 10
    'lr_all': [0.002, 0.005],# 📈 Tasa de aprendizaje para todos los parámetros (qué tan rápido ajusta el modelo)
    'reg_all': [0.4, 0.6]    # ⚖️ Factor de regularización (controla el sobreajuste) a probar: 0.4 o 0.6
}

In [19]:
gs.fit(dataset)

Imprima el rmse y el fcp, y la mejor combinación de parámetros

In [20]:
gs.best_score

{'fcp': 0.6993020842452041, 'rmse': 0.963905867924563}

In [ ]:
gs.best_params

{
    'fcp': {                     # 📊 Mejores parámetros encontrados para la métrica FCP (Fraction of Concordant Pairs)
        'n_epochs': 10,          # ⏳ Número de épocas = 10
        'lr_all': 0.005,         # 📈 Tasa de aprendizaje = 0.005
        'reg_all': 0.6           # ⚖️ Regularización = 0.6
    },
    'rmse': {                    # 📉 Mejores parámetros encontrados para la métrica RMSE (Root Mean Squared Error)
        'n_epochs': 10,          # ⏳ Número de épocas = 10
        'lr_all': 0.005,         # 📈 Tasa de aprendizaje = 0.005
        'reg_all': 0.4           # ⚖️ Regularización = 0.4
    }
}


{'fcp': {'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0.6},
 'rmse': {'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0.4}}

Guarde el modelo con mayor fcp y prediga el rating para el user id 196 e item id 242

In [22]:
best_model = gs.best_estimator["fcp"]

In [23]:
pred = best_model.predict("196", "242")
pred

Prediction(uid='196', iid='242', r_ui=None, est=3.52986, details={'was_impossible': False})

In [24]:
pred.est

3.52986

Pruebe empleando otros modelos como SVDpp,  NMF,  KNNWithZScore e intente superar el valor obtenido

In [25]:
from surprise import SVDpp

'''
class surprise.prediction_algorithms.matrix_factorization.SVDpp(n_factors=20, n_epochs=20,
init_mean=0, init_std_dev=0.1, lr_all=0.007, reg_all=0.02, lr_bu=None, lr_bi=None,
lr_pu=None, lr_qi=None, lr_yj=None, reg_bu=None, reg_bi=None, reg_pu=None, reg_qi=None,
reg_yj=None, random_state=None, verbose=False, cache_ratings=False)
'''

gs1 = GridSearchCV(SVDpp, param_grid, measures=['fcp',"rmse"], cv=3, refit=True)
gs1.fit(dataset)
gs1.best_score

{'fcp': 0.6985723171062354, 'rmse': 0.963834874126817}

In [ ]:
from surprise import NMF

'''
class surprise.prediction_algorithms.matrix_factorization.NMF(n_factors=15, n_epochs=50,
biased=False, reg_pu=0.06, reg_qi=0.06, reg_bu=0.02, reg_bi=0.02, lr_bu=0.005,
lr_bi=0.005, init_low=0, init_high=1, random_state=None, verbose=False)
'''

param_grid2 = {
    'n_epochs': [5, 10],     # ⏳ número de iteraciones: 5 o 10
    'lr_bu': [0.002, 0.005], # 📈 tasa de aprendizaje de sesgo de usuarios
    'lr_bi': [0.002, 0.005], # 📈 tasa de aprendizaje de sesgo de ítems
    'reg_pu': [0.04, 0.06],  # ⚖️ regularización de factores de usuario
    'reg_qi': [0.04, 0.06],  # ⚖️ regularización de factores de ítem
    'reg_bu': [0.01, 0.03],  # ⚖️ regularización del sesgo de usuario
    'reg_bi': [0.01, 0.03]   # ⚖️ regularización del sesgo de ítem
}


# 🔹 Configuramos GridSearchCV de Surprise:
gs2 = GridSearchCV(NMF, param_grid2, measures=['fcp', 'rmse'], cv=3, refit=True)

# 🔹 Entrenamos la búsqueda de hiperparámetros con el dataset
gs2.fit(dataset)

# 🔹 Obtenemos los mejores scores alcanzados en cada métrica
gs2.best_score

{'fcp': 0.6807268770144659, 'rmse': 0.9994562283774995}

In [27]:
from surprise import KNNWithZScore

'''
classsurprise.prediction_algorithms.knns.KNNWithZScore(k=40, min_k=1,
sim_options={}, verbose=True, **kwargs)
'''
param_grid3={'k': range(20, 65,5)}
gs3 = GridSearchCV(KNNWithZScore, param_grid3, measures=['fcp',"rmse"], cv=3, refit=True)
gs3.fit(dataset)
gs3.best_score

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

{'fcp': 0.705360284098843, 'rmse': 0.9558750271831403}

# Recomendación basada en el contenido

En este ejemplo vamos a tomar un corpus de textos de autores latinoamericanos para sugerir uno similar a uno dado. Para esto construiremos una matriz TFIDF, de frecuencias normalizadas de términos por documento, y usaremos la similitud coseno para medir distancias entre los distintos textos.

In [28]:
!git clone https://github.com/karen-pal/borges

Cloning into 'borges'...
remote: Enumerating objects: 328, done.
remote: Counting objects: 100% (328/328), done.
remote: Compressing objects: 100% (264/264), done.
remote: Total 328 (delta 131), reused 249 (delta 60), pack-reused 0 (from 0)
Receiving objects: 100% (328/328), 26.76 MiB | 7.73 MiB/s, done.
Resolving deltas: 100% (131/131), done.


In [29]:
!ls


1 - Presentación Sistemas de Recomendación.pdf
Sistemas_Recomendacion_Github.ipynb
borges
sistemas_recomendacion_surprise.ipynb


In [30]:
!cd borges
!ls borges/

LDA.ipynb                 datasets                  scraper.py
README.md                 full_text_scrapper.py     sentence_similarity.ipynb
borges.ipynb              link_scrapper.py          uso_simple.ipynb


In [31]:
!ls borges/datasets

borges_sentiment_corpus.csv full_corpus.csv
datasets_csv                links
datasets_pkl                pkl_to_csv.ipynb


In [32]:
import pickle
from pathlib import Path
import pandas as pd

df = pd.DataFrame()
# usando el asterisco de "wildcard" traemos todos los archivos en formato pickle
pkls = Path('.').glob('./borges/datasets/datasets_pkl/*texts.pkl')

# leemos todos los pickles y concatenarlos en un DataFrame
for pkl in pkls:
    with open(pkl, 'rb') as inp:
        df_ = pickle.load(inp)
    df = pd.concat([df, df_])

df.shape

(719, 3)

In [33]:
df

,link,text_metadata,text
0,https://ciudadseva.com/texto/la-hija-del-guard...,"{'title': 'La hija del guardaagujas', 'metadat...",La casita del guardaagujas está junto a la lín...
1,https://ciudadseva.com/texto/la-joven-del-abri...,"{'title': 'La joven del abrigo largo', 'metada...",Cruza todos los días la plaza en el mismo sent...
2,https://ciudadseva.com/texto/tragedia/,"{'title': 'Tragedia', 'metadata': '[Minicuento...",María Olga es una mujer encantadora. Especialm...
0,https://ciudadseva.com/texto/azogue/,"{'title': 'Azogue', 'metadata': '[Minicuento -...",Pobrecita Alicia. Aunque la razón te decía no ...
1,https://ciudadseva.com/texto/casa-di-amore-e-p...,"{'title': 'Casa di Amore e Psyche', 'metadata'...","Pero qué maravilla, qué maravilla, decía la se..."
...,...,...,...
0,https://ciudadseva.com/texto/en-el-insomnio/,"{'title': 'En el insomnio', 'metadata': '[Mini...",El hombre se acuesta temprano. No puede concil...
1,https://ciudadseva.com/texto/la-carne/,"{'title': 'La carne', 'metadata': '[Cuento - T...","Sucedió con gran sencillez, sin afectación. Po..."
2,https://ciudadseva.com/texto/natacion/,"{'title': 'Natación', 'metadata': '[Minicuento...",He aprendido a nadar en seco. Resulta más vent...
0,https://ciudadseva.com/texto/el-tren-dabove/,"{'title': 'El tren', 'metadata': '[Cuento - Te...","El tren era todos los días a la tardecita, per..."


In [34]:
df['text_metadata'].sample(2)

15    {'title': 'Nube de polillas', 'metadata': '[Mi...
17    {'title': 'La gloriosa', 'metadata': '[Minicue...
Name: text_metadata, dtype: object

In [35]:
df.columns

Index(['link', 'text_metadata', 'text'], dtype='object')

In [36]:
# separamos de la metadata el título y autor en sus propias columnas
df['title'] = df['text_metadata'].apply(lambda x: x['title'])
df['author'] = df['text_metadata'].apply(lambda x: x['author'])

In [37]:
# vemos los autores disponibloes
df['author'].value_counts()

author
Jorge Luis Borges             60
Julio Cortázar                55
Baldomero Lillo               50
Juan José Arreola             45
Augusto Monterroso            45
Alfonso Reyes                 37
Enrique Anderson Imbert       36
Mario Benedetti               33
Julio Ramón Ribeyro           27
Roberto Arlt                  25
Clarice Lispector             25
Julio Torri                   23
Felisberto Hernández          15
Luis Vidales                  14
Adolfo Bioy Casares           13
Rubén Darío                   13
Álvaro Mutis                  11
Juan Rulfo                    10
Juan Rodolfo Wilcock          10
Edmundo Valadés               10
Elena Garro                    9
Manuel A. Alonso               9
Salarrué                       9
Juan Bosch                     8
Eduardo Gudiño Kieffer         8
Alejo Carpentier               8
Silvina Ocampo                 7
Virgilio Díaz Grullón          7
Andrés Rivera                  7
Rodolfo Walsh                  6
Ric

In [38]:
# quitamos duplicados y reiniciamos el índice
df = df.drop_duplicates(subset=[c for c in df.columns if c != 'text_metadata'])
df = df.reset_index(drop=True)
df.shape

(693, 5)

In [39]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel
from pprint import pprint

Vamos a calcular las matrices de ocurrencias de términos usando sklearn.

Ámbas clases primero construyen el vocabulario total, y luego:  
- **CountVectorizer** nos devuelve la frecuencia absoluta de cada término por cada documento.
- [**TF-IDF**](https://en.wikipedia.org/wiki/Tf%E2%80%93idf): calcula la frecuencia de cada término por documento, y normaliza por el total de documentos donde el término aparece.

$${tf} (t,d)={\frac {f_{t,d}}{\sum _{t'\in d}{f_{t',d}}}}$$

$$
idf( t, D ) = log \frac{ \text{| } D \text{ |} }{ 1 + \text{| } \{ d \in D : t \in d \} \text{ |} }
$$


$$ tfidf( t, d, D ) = tf( t, d ) \times idf( t, D )
$$


In [ ]:
# Instanciamos el CV
vectorizer = CountVectorizer()

doc1 = 'la matriz de frecuencias por palabras otorga información del contenido de un documento'
doc2 = 'las palabras que aparecen en un documento se relaciona con su tema'
# Definimos una lista con todos los strings
data_corpus = [doc1, doc2]

# Fiteamos el CV y transformamos los datos
X = vectorizer.fit_transform(data_corpus)
# 👉 fit_transform hace dos cosas:
# Aprende el vocabulario (palabras únicas en todos los documentos).
# Transforma cada documento en un vector con las frecuencias de esas palabras.
# El resultado X es una sparse matrix (matriz dispersa), eficiente en memoria.

# Pasamos de sparse matrix a array usando .toarray()

print(X.toarray())
# Usando el metodo .get_feature_names() del CV podemos acceder al indice de palabras
print(vectorizer.get_feature_names_out())

[[0 0 1 2 1 1 0 1 1 1 0 1 1 1 1 0 0 0 0 0 1]
 [1 1 0 0 0 1 1 0 0 0 1 0 0 1 0 1 1 1 1 1 1]]
['aparecen' 'con' 'contenido' 'de' 'del' 'documento' 'en' 'frecuencias'
 'información' 'la' 'las' 'matriz' 'otorga' 'palabras' 'por' 'que'
 'relaciona' 'se' 'su' 'tema' 'un']


In [41]:
X

<2x21 sparse matrix of type '<class 'numpy.int64'>'
	with 24 stored elements in Compressed Sparse Row format>

In [42]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop = list(stopwords.words('spanish'))
# eliminamos las "stop words", palabras comunes no informativas
tf = TfidfVectorizer(stop_words=stop)

[nltk_data] Downloading package stopwords to /Users/abc/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [43]:
# calculamos los features para cada ítem (texto)
tfidf_matrix = tf.fit_transform(df['text'])

In [44]:
tfidf_matrix.shape

(693, 56608)

In [45]:
# calculamos las similitudes entre todos los documentos
cosine_similarities = linear_kernel(tfidf_matrix, tfidf_matrix)
n = 6

# diccionario creado para guardar el resultado en un formato (autor - titulo : puntaje, titulo, autor)
results = {}
for idx, row in df.iterrows():
    # guardamos los indices similares basados en la similitud coseno. Los ordenamos en modo ascendente, siendo 0 nada de similitud y 1 total
    similar_indices = cosine_similarities[idx].argsort()[:-n-2:-1]
    # guardamos los N más cercanos
    similar_items = [(f"{df['author'][i]} - {df['title'][i]}", round(cosine_similarities[idx][i], 3)) for i in similar_indices]
    results[f"{row['author']} - {row['title']}"] = similar_items[1:]

In [46]:
pprint(results['Jorge Luis Borges - El Aleph'])

[('Jorge Luis Borges - La escritura del dios', 0.144),
 ('Jorge Luis Borges - El inmortal', 0.135),
 ('Jorge Luis Borges - Utopía de un hombre que está cansado', 0.125),
 ('Felisberto Hernández - El acomodador', 0.122),
 ('Clarice Lispector - La búsqueda de la dignidad', 0.121),
 ('Jorge Luis Borges - Funes el memorioso', 0.11)]


In [47]:
def recomendar(autor, titulo):
    pprint(results[f"{autor} - {titulo}"])

In [48]:
recomendar('Julio Cortázar', 'Axolotl')

[('Felisberto Hernández - El acomodador', 0.134),
 ('Felisberto Hernández - El cocodrilo', 0.101),
 ('Felisberto Hernández - Menos Julia', 0.089),
 ('Julio Cortázar - Después del almuerzo', 0.088),
 ('Julio Cortázar - La noche boca arriba', 0.086),
 ('Julio Cortázar - La señorita Cora', 0.086)]


In [49]:
# https://medium.com/@eng.saavedra/sistemas-de-recomendaci%C3%B3n-parte-2-b8a5dc9dc730
# https://towardsdatascience.com/item-based-collaborative-filtering-in-python-91f747200fab
# https://www.genbeta.com/web/asi-funcionan-las-recomendaciones-de-amazon
